In [15]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import norm

np.random.seed(42)

In [16]:
# 1
tickers = ["MSFT", "AAPL", "GOOGL"]
prices = yf.download(
    tickers,
    start="2024-05-23",
    end="2026-05-23",
    auto_adjust=False,
    progress=False
)["Close"]
prices = prices[tickers]

print("First five observations:")
display(prices.head())

print("Last five observations:")
display(prices.tail())

print("Total number of observations:", len(prices))


First five observations:


Ticker,MSFT,AAPL,GOOGL
Date,,,
2024-05-23,427.000000,186.880005,173.550003
2024-05-24,430.160004,189.979996,174.990005
2024-05-28,430.320007,189.990005,176.399994
2024-05-29,429.170013,190.289993,175.899994
2024-05-30,414.670013,191.289993,172.110001


Last five observations:


Ticker,MSFT,AAPL,GOOGL
Date,,,
2026-05-18,423.540009,297.839996,396.940002
2026-05-19,417.420013,298.970001,387.660004
2026-05-20,421.059998,302.250000,388.910004
2026-05-21,419.089996,304.989990,387.660004
2026-05-22,418.570007,308.820007,382.970001


Total number of observations: 501


In [17]:
#2
market_caps = pd.Series({
    "MSFT": 3.109,
    "AAPL": 4.536,
    "GOOGL": 4.640
})

portfolio_value = 1_000_000

print("Sum of market caps:", market_caps.sum())
weights = market_caps / market_caps.sum()
dollar_positions = portfolio_value * weights

print("Market-cap weights:")
display(weights)

print("Dollar positions:")
display(dollar_positions)

Sum of market caps: 12.285
Market-cap weights:


MSFT     0.253073
AAPL     0.369231
GOOGL    0.377696
dtype: float64

Dollar positions:


MSFT     253072.853073
AAPL     369230.769231
GOOGL    377696.377696
dtype: float64

In [18]:
def empirical_var(losses, alpha):
    """
    Compute empirical VaR using the lecture-note definition:
    VaR_alpha = l_(ceil(n * alpha)).
    """
    sorted_losses = np.sort(losses)
    n = len(sorted_losses)

    # Convert mathematical 1-based index to Python 0-based index
    index = int(np.ceil(n * alpha)) - 1

    return sorted_losses[index]

In [19]:
#3
alpha = 0.95

log_returns = np.log(prices / prices.shift(1)).dropna()
X = log_returns[["MSFT", "AAPL", "GOOGL"]].to_numpy()
w = weights[["MSFT", "AAPL", "GOOGL"]].to_numpy()
V = portfolio_value
full_losses = -V * np.sum(
    w * (np.exp(X) - 1),
    axis=1
)
linear_losses = -V * np.sum(
    w * X,
    axis=1
)
quadratic_losses = -V * np.sum(
    w * (X + 0.5 * X**2),
    axis=1
)

var_full = empirical_var(full_losses, alpha)
var_linear = empirical_var(linear_losses, alpha)
var_quadratic = empirical_var(quadratic_losses, alpha)

var_results = pd.DataFrame({
    "Loss Operator": ["Full", "Linear", "Quadratic"],
    "VaR 95%": [var_full, var_linear, var_quadratic]
})

display(var_results)

,Loss Operator,VaR 95%
0,Full,21043.046108
1,Linear,21415.726274
2,Quadratic,21038.613973


In [20]:
#4
lambda_ewma = 0.97
theta_ewma = 0.97
 
X = log_returns.sort_index().to_numpy()

mu = np.zeros(3)
Sigma = np.zeros((3, 3))

for x in X:
    deviation = x - mu

    Sigma = (
        theta_ewma * Sigma
        + (1 - theta_ewma) * np.outer(deviation, deviation)
    )

    mu = (
        lambda_ewma * mu
        + (1 - lambda_ewma) * x
    )

mu_ewma = pd.Series(mu, index=tickers, name="EWMA Mean")

Sigma_ewma = pd.DataFrame(
    Sigma,
    index=tickers,
    columns=tickers
)

display(mu_ewma)
display(Sigma_ewma)



MSFT     0.000473
AAPL     0.003636
GOOGL    0.003306
Name: EWMA Mean, dtype: float64

,MSFT,AAPL,GOOGL
MSFT,0.000314,0.000034,0.000003
AAPL,0.000034,0.000193,0.000081
GOOGL,0.000003,0.000081,0.000459


In [21]:
#5
#Linear Loss
linear_loss_mean = -dollar_positions @ mu

linear_loss_std = np.sqrt(
    dollar_positions @ Sigma @ dollar_positions
)

var_linear_normal = (
    linear_loss_mean
    + linear_loss_std * norm.ppf(alpha)
)

# Full Loss and Quadratic Loss(MC)
N = 100_000
X_sim = np.random.multivariate_normal(
    mean=mu,
    cov=Sigma,
    size=N
)
full_losses_mc = -V * np.sum(
    w * (np.exp(X_sim) - 1),
    axis=1
)
quadratic_losses_mc = -V * np.sum(
    w * (X_sim + 0.5 * X_sim**2),
    axis=1
)
var_full_normal = empirical_var(
    full_losses_mc,
    alpha
)

var_quadratic_normal = empirical_var(
    quadratic_losses_mc,
    alpha
)

var_normal_results = pd.DataFrame({
    "Loss Operator": [
        "Full (MC)",
        "Linear (Analytical)",
        "Quadratic (MC)"
    ],
    "VaR 95%": [
        var_full_normal,
        var_linear_normal,
        var_quadratic_normal
    ]
})

display(var_normal_results)

,Loss Operator,VaR 95%
0,Full (MC),16513.315200
1,Linear (Analytical),16869.110890
2,Quadratic (MC),16512.217126
